2. Identify customer records with missing Customer_Name, Mobile_Number, or KYC_Status. Report missing counts by field and list affected customers.

In [0]:
from pyspark.sql.functions import col, count, when
df = spark.table("banking.dq.silver_customers")

cust_name = df.filter(col("Customer_Name").isNull()).alias("Missing customer Name")
mob_no = df.filter(col("Mobile_Number").isNull()).alias("Missing Mobile Number")
miss_kyc= df.filter(col("KYC_Status").isNull()).alias("Missing KYC")
display(cust_name)
display(mob_no)
display(miss_kyc)

print("Missing Customer Name:", cust_name.count())
print("Missing Mobile Number:", mob_no.count())
print("Missing KYC:", miss_kyc.count())

affected = df.filter(
    col("Customer_Name").isNull() |
    col("Mobile_Number").isNull() |
    col("KYC_Status").isNull()
)
print("Affected customers:", affected.count())
display(affected)


Customer_ID,Customer_Name,City,Mobile_Number,KYC_Status
C00018,null,Hyderabad,9000000018,Pending


Customer_ID,Customer_Name,City,Mobile_Number,KYC_Status
C00047,Neha Gupta,Mumbai,null,Verified
C00094,Arjun Mehta,Mumbai,null,Verified
C00141,Priya Reddy,Chennai,null,Verified
C00188,Vikram Singh,Bengaluru,null,Verified
C00235,Ananya Iyer,Hyderabad,null,Verified
C00282,Amit Kumar,Hyderabad,null,Verified
C00329,Farah Khan,Kolkata,null,Verified
C00376,Kiran Patel,Bengaluru,null,Verified
C00423,Sneha Rao,Hyderabad,null,Pending
C00470,Rahul Sharma,Delhi,null,Verified


Customer_ID,Customer_Name,City,Mobile_Number,KYC_Status
C00083,Sneha Rao,Hyderabad,9000000083,null
C00166,Kiran Patel,Delhi,9000000166,null
C00249,Farah Khan,Mumbai,9000000249,null
C00332,Amit Kumar,Mumbai,9000000332,null
C00415,Ananya Iyer,Chennai,9000000415,null
C00498,Vikram Singh,Chennai,9000000498,null
C00581,Priya Reddy,Hyderabad,9000000581,null
C00664,Arjun Mehta,Bengaluru,9000000664,null
C00747,Neha Gupta,Kolkata,9000000747,null
C00830,Rahul Sharma,Pune,9000000830,null


Missing Customer Name: 1
Missing Mobile Number: 21
Missing KYC: 12
Affected customers: 34


Customer_ID,Customer_Name,City,Mobile_Number,KYC_Status
C00018,null,Hyderabad,9000000018,Pending
C00047,Neha Gupta,Mumbai,null,Verified
C00083,Sneha Rao,Hyderabad,9000000083,null
C00094,Arjun Mehta,Mumbai,null,Verified
C00141,Priya Reddy,Chennai,null,Verified
C00166,Kiran Patel,Delhi,9000000166,null
C00188,Vikram Singh,Bengaluru,null,Verified
C00235,Ananya Iyer,Hyderabad,null,Verified
C00249,Farah Khan,Mumbai,9000000249,null
C00282,Amit Kumar,Hyderabad,null,Verified


3. Identify accounts whose Customer_ID is not present in the customer dataset. Return the account details and the number of affected accounts.

In [0]:
acc = spark.table("banking.dq.silver_accounts")
cust = spark.table("banking.dq.silver_customers")

orphan_acc = acc.join(cust, on="Customer_ID", how="left_anti")
display(orphan_acc)

print("Affected accounts:", orphan_acc.count())


Customer_ID,Account_ID,Account_Type,Balance,Status
C09137,A00137,Current,185717.26,Active
C09274,A00274,Current,219241.42,Active
C09411,A00411,Savings,339104.27,Active
C09548,A00548,Savings,233648.10,Active
C09685,A00685,Savings,432953.86,Active
C09822,A00822,Savings,52673.17,Active
C09959,A00959,Current,225721.51,Active
C10096,A01096,Current,143118.89,Active
C10233,A01233,Savings,117112.11,Active
C10370,A01370,Savings,221843.75,Active


Affected accounts: 10


4. Find transaction IDs present in source_transactions but absent from target_transactions. Return their source details and total count.

In [0]:
src = spark.table("banking.dq.silver_source_transactions")
tgt = spark.table("banking.dq.silver_target_transactions")
TrID_insrc_notintgt = src.join(tgt, on="Transaction_ID", how="left_anti")
display(TrID_insrc_notintgt)
print("Missing in target:", TrID_insrc_notintgt.count())


Transaction_ID,Account_ID,Transaction_Date,Amount,Type,Status
T000007,A01392,2026-10-27,32385.77,UPI,Success
T000036,A00301,2026-10-02,80177.25,Debit,Success
T000059,A01142,2026-10-14,49711.84,NEFT,Success
T000074,A00323,2026-10-08,58968.97,Debit,Success
T000085,A00057,2026-10-29,52835.05,NEFT,Success
T000188,A01348,2026-10-21,75275.14,NEFT,Success
T000199,A01127,2026-10-06,71331.92,NEFT,Success
T000214,A01413,2026-10-06,43970.46,IMPS,Success
T000248,A01085,2026-10-02,22302.92,UPI,Success
T000291,A00367,2026-10-06,78200.23,Debit,Success


Missing in target: 260


5. Find transaction IDs present in target_transactions but absent from source_transactions. Return their target details and total count.

In [0]:
src = spark.table("banking.dq.silver_source_transactions")
tgt = spark.table("banking.dq.silver_target_transactions")
TrID_intgt_notinsrc = tgt.join(src, on="Transaction_ID", how="left_anti")
display(TrID_intgt_notinsrc)
print("Missing in source/extra target:", TrID_intgt_notinsrc.count())

Transaction_ID,Account_ID,Transaction_Date,Amount,Type,Status
TX000003,A00003,2026-10-15,100.00,Debit,Success
TX000001,A00001,2026-10-15,100.00,Debit,Success
TX000002,A00002,2026-10-15,100.00,Debit,Success
TX000005,A00005,2026-10-15,100.00,Debit,Success
TX000004,A00004,2026-10-15,100.00,Debit,Success


Missing in source/extra target: 5


6. Identify duplicate Transaction_ID values in target_transactions. Report each duplicated ID and its occurrence count.

In [0]:
tgt = spark.table("banking.dq.silver_target_transactions")

dup_TID= tgt.groupBy("Transaction_ID").agg(count("Transaction_ID").alias("occurrence count")).filter(col("occurrence count") > 1)
display(dup_TID)

print("Duplicate Transaction ID:", dup_TID.count())

Transaction_ID,occurrence count
T000421,2
T000434,2
T000432,2
T000422,2
T000426,2
T000425,2
T000429,2
T000423,2
T000419,2
T000430,2


Duplicate Transaction ID: 15


7. Find target transactions where Amount is missing, zero, or negative. Report counts for each condition and the affected transactions.

In [0]:
tgt = spark.table("banking.dq.silver_target_transactions")
Amt_missing= tgt.filter(col("Amount").isNull())

print("Amount missing:" , Amt_missing.count())
display(Amt_missing)

Amt_zero= tgt.filter(col("Amount") == 0)
print("Amount is zero :" , Amt_zero.count())
display(Amt_zero)



Amt_neg= tgt.filter(col("Amount") < 0)
print("Amount is negative :" , Amt_neg.count())
display(Amt_neg)


affected_txn = tgt.filter(col("Amount").isNull() | (col("Amount") <= 0))
print("Affected transactions:", affected_txn.count())
display(affected_txn)



Amount missing: 20


Transaction_ID,Account_ID,Transaction_Date,Amount,Type,Status
T003105,A00351,2026-10-07,null,Debit,Success
T004652,A00008,2026-10-30,null,Credit,Success
T006710,A01462,2026-10-25,null,NEFT,Failed
T002590,A00756,2026-10-25,null,Credit,Success
T008262,A00706,2026-10-07,null,UPI,Success
T006192,A01394,2026-10-20,null,IMPS,Success
T001564,A00261,2026-10-19,null,IMPS,Success
T002077,A00173,2026-10-08,null,NEFT,Success
T005678,A01167,2026-10-06,null,UPI,Success
T003622,A01036,2026-10-26,null,Credit,Success


Amount is zero : 9


Transaction_ID,Account_ID,Transaction_Date,Amount,Type,Status
T006893,A00303,2026-10-07,0.00,UPI,Success
T004638,A00001,2026-10-04,0.00,Debit,Success
T009167,A00676,2026-10-12,0.00,UPI,Success
T000107,A00269,2026-10-15,0.00,IMPS,Success
T001252,A01300,2026-10-28,0.00,Credit,Success
T008031,A00556,2026-10-24,0.00,Debit,Success
T003508,A01075,2026-10-12,0.00,Debit,Success
T002377,A00679,2026-10-18,0.00,Credit,Success
T005763,A00618,2026-10-16,0.00,NEFT,Success


Amount is negative : 5


Transaction_ID,Account_ID,Transaction_Date,Amount,Type,Status
T000312,A00266,2026-10-20,-15.00,IMPS,Failed
T002271,A00213,2026-10-12,-15.00,Credit,Success
T004218,A00275,2026-10-11,-15.00,Credit,Failed
T006157,A00621,2026-10-22,-15.00,IMPS,Success
T008118,A01074,2026-10-22,-15.00,UPI,Success


Affected transactions: 34


Transaction_ID,Account_ID,Transaction_Date,Amount,Type,Status
T003105,A00351,2026-10-07,null,Debit,Success
T006893,A00303,2026-10-07,0.00,UPI,Success
T000312,A00266,2026-10-20,-15.00,IMPS,Failed
T002271,A00213,2026-10-12,-15.00,Credit,Success
T004638,A00001,2026-10-04,0.00,Debit,Success
T004652,A00008,2026-10-30,null,Credit,Success
T006710,A01462,2026-10-25,null,NEFT,Failed
T004218,A00275,2026-10-11,-15.00,Credit,Failed
T002590,A00756,2026-10-25,null,Credit,Success
T006157,A00621,2026-10-22,-15.00,IMPS,Success


8. Compare source and target transaction row counts for each Transaction_Date. Show both counts and their difference for every date where they differ.

In [0]:
from pyspark.sql.functions import col, count


src_cnt = src.groupBy("Transaction_Date").agg(count("*").alias("source_count"))


tgt_cnt = tgt.groupBy("Transaction_Date").agg(count("*").alias("target_count"))

both_cmp = src_cnt.join(tgt_cnt, on="Transaction_Date", how="full_outer").fillna(0)


both_cmp = both_cmp.withColumn("difference", col("source_count") - col("target_count"))

diff_dates = both_cmp.filter(col("difference") != 0).orderBy("Transaction_Date")
display(diff_dates)

Transaction_Date,source_count,target_count,difference
2026-10-01,330,326,4
2026-10-02,312,305,7
2026-10-03,306,298,8
2026-10-04,326,318,8
2026-10-05,330,320,10
2026-10-06,341,331,10
2026-10-07,320,313,7
2026-10-08,284,268,16
2026-10-09,317,315,2
2026-10-10,313,306,7


9. For matching Transaction_ID values, identify records whose Amount differs between source and target, including cases where either Amount is missing. Report the source and target amounts.

In [0]:
src = spark.table("banking.dq.silver_source_transactions")
tgt = spark.table("banking.dq.silver_target_transactions")

src= src.select("Transaction_ID",col("Amount").alias("source_amount"))
tgt= tgt.select("Transaction_ID",col("Amount").alias("target_amount"))
matched= src.join(tgt,on="Transaction_ID",how="inner")

amt_diff= matched.filter((col("source_amount") != col("target_amount")) | col("source_amount").isNull() | col("target_amount").isNull())
display(amt_diff)
print("Amount mismatch:", amt_diff.count())

Transaction_ID,source_amount,target_amount
T003105,66702.46,null
T005959,81368.20,81375.20
T006893,7851.99,0.00
T000312,31879.11,-15.00
T002271,3506.78,-15.00
T007397,12158.67,12165.67
T004638,22660.03,0.00
T004652,72248.82,null
T006710,65999.10,null
T004218,72655.84,-15.00


Amount mismatch: 48


10. Create a final data quality summary showing the results of checks 2–9, and save the exception details and summary as Delta tables.

In [0]:
from pyspark.sql.functions import col, lit, concat_ws

def exc(df, check_no, check_name, id_col):
    return df.select(
        lit(check_no).alias("check_no"),
        lit(check_name).alias("check_name"),
        col(id_col).cast("string").alias("record_id"),
        concat_ws(" | ", *[col(c).cast("string") for c in df.columns]).alias("details")
    )

exceptions = (
    exc(affected, 2, "Customer missing/invalid fields", "Customer_ID")
    .unionByName(exc(orphan_acc, 3, "Account with no customer", "Account_ID"))
    .unionByName(exc(TrID_insrc_notintgt, 4, "Txn missing in target", "Transaction_ID"))
    .unionByName(exc(TrID_intgt_notinsrc, 5, "Extra txn in target", "Transaction_ID"))
    .unionByName(exc(dup_TID, 6, "Duplicate txn in target", "Transaction_ID"))
    .unionByName(exc(affected_txn, 7, "Amount missing/zero/negative", "Transaction_ID"))
    .unionByName(exc(diff_dates, 8, "Daily count mismatch", "Transaction_Date"))
    .unionByName(exc(amt_diff, 9, "Amount mismatch src vs tgt", "Transaction_ID"))
)
display(exceptions)

check_no,check_name,record_id,details
2,Customer missing/invalid fields,C00018,C00018 | Hyderabad | 9000000018 | Pending
2,Customer missing/invalid fields,C00047,C00047 | Neha Gupta | Mumbai | Verified
2,Customer missing/invalid fields,C00083,C00083 | Sneha Rao | Hyderabad | 9000000083
2,Customer missing/invalid fields,C00094,C00094 | Arjun Mehta | Mumbai | Verified
2,Customer missing/invalid fields,C00141,C00141 | Priya Reddy | Chennai | Verified
2,Customer missing/invalid fields,C00166,C00166 | Kiran Patel | Delhi | 9000000166
2,Customer missing/invalid fields,C00188,C00188 | Vikram Singh | Bengaluru | Verified
2,Customer missing/invalid fields,C00235,C00235 | Ananya Iyer | Hyderabad | Verified
2,Customer missing/invalid fields,C00249,C00249 | Farah Khan | Mumbai | 9000000249
2,Customer missing/invalid fields,C00282,C00282 | Amit Kumar | Hyderabad | Verified


In [0]:
from pyspark.sql.functions import count

summary = (exceptions.groupBy("check_no", "check_name")
           .agg(count("*").alias("failed_count"))
           .orderBy("check_no"))
display(summary)

exceptions.write.format("delta").mode("overwrite").saveAsTable("banking.dq.gold_dq_exceptions")
summary.write.format("delta").mode("overwrite").saveAsTable("banking.dq.gold_dq_summary")
display(spark.table("banking.dq.gold_dq_summary"))

check_no,check_name,failed_count
2,Customer missing/invalid fields,34
3,Account with no customer,10
4,Txn missing in target,260
5,Extra txn in target,5
6,Duplicate txn in target,15
7,Amount missing/zero/negative,34
8,Daily count mismatch,31
9,Amount mismatch src vs tgt,48


check_no,check_name,failed_count
2,Customer missing/invalid fields,34
3,Account with no customer,10
4,Txn missing in target,260
5,Extra txn in target,5
6,Duplicate txn in target,15
7,Amount missing/zero/negative,34
8,Daily count mismatch,31
9,Amount mismatch src vs tgt,48
